# 03 — Rotary Position Embedding (M1.3)

How Qwen3 tells attention *where* each token is. This notebook imports the tested
functions from `qwen3_scratch.layers`. CPU only, a few seconds, no downloads. Plots
need `matplotlib`; without it, text tables are printed.

Full write-up: `docs/components/rope.md`.

## 1. The problem: attention has no notion of order

Self-attention scores $q_m\cdot k_n$ do not depend on where tokens are: permute the
input and the outputs permute the same way. Causal masking adds *some* order
information, but the model needs explicit positions.

| approach | how | drawback |
|---|---|---|
| learned absolute (GPT-2) | add a trained vector $p_m$ to the embedding | no positions beyond training length |
| sinusoidal absolute (Transformer 2017) | add fixed $\sin/\cos$ of $m\omega_i$ | relative distance only implicit |
| relative (T5, Shaw et al.) | bias/embedding depending on $n-m$ inside attention | extra parameters / compute |
| **RoPE** (Su et al. 2021) | **rotate** $q$ and $k$ by angle $\propto$ position | extrapolation still limited |

RoPE makes the score depend on the *relative* offset $n-m$ with no extra
parameters and no change to the attention formula.

## 2. The 2D rotation

$$R(\theta)=\begin{pmatrix}\cos\theta&-\sin\theta\\ \sin\theta&\cos\theta\end{pmatrix},\qquad
R(\alpha)^\top R(\beta)=R(\beta-\alpha)$$

So $\langle R(m\omega)q,\;R(n\omega)k\rangle = q^\top R((n-m)\omega)\,k$: the
score sees only $n-m$. A head vector of size $D$ is treated as $D/2$ independent
2D pairs, pair $i$ rotating with frequency $\omega_i=\theta^{-2i/D}$.

In [ ]:
import math

import torch

from qwen3_scratch.layers import (
    RotaryEmbedding,
    apply_rope,
    apply_rope_to_qk,
    compute_inv_freq,
    make_position_ids,
    rope_cos_sin,
    rotate_half,
)
from qwen3_scratch.layers.rope import apply_rope_interleaved, half_layout_permutation, rotate_2d

v = torch.tensor([1.0, 0.5], dtype=torch.float64)
omega = 0.5  # rad per token, for illustration
points = [rotate_2d(v, p * omega) for p in range(6)]
for p, pt in enumerate(points):
    xy = f"({pt[0]:+.3f}, {pt[1]:+.3f})"
    print(f"position {p}: angle {p * omega:.1f} rad -> {xy}, norm {pt.norm():.4f}")

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is not None:
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    for p, pt in enumerate(points):
        ax.annotate(
            "",
            xy=(pt[0], pt[1]),
            xytext=(0, 0),
            arrowprops={"arrowstyle": "->", "lw": 2, "color": "#2a78d6"},
        )
        ax.text(pt[0] * 1.12, pt[1] * 1.12, f"p={p}", ha="center", va="center", fontsize=9)
    circle = plt.Circle((0, 0), v.norm().item(), fill=False, ls="--", lw=0.8, color="#9a9a94")
    ax.add_patch(circle)
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect("equal")
    ax.grid(alpha=0.25, lw=0.6)
    ax.set_title("One feature pair rotated by p·ω (ω = 0.5)")
    plt.show()

## 3. Frequencies, theta and the cos/sin tables

$\omega_i=\theta^{-2i/D}$, $i=0..D/2-1$. Pair 0 turns 1 rad per token
(wavelength $2\pi$); the last pair barely moves. A larger $\theta$ stretches the
slow end, giving distinguishable angles over longer contexts. Qwen3-0.6B uses
$D=128$, $\theta=10^6$.

In [ ]:
from qwen3_scratch.config import QWEN3_0_6B_BASE as CFG

inv_1e4 = compute_inv_freq(CFG.head_dim, 10_000.0, dtype=torch.float64)
inv_1e6 = compute_inv_freq(CFG.head_dim, CFG.rope_theta, dtype=torch.float64)
for i in (0, 16, 32, 48, 63):
    print(
        f"pair {i:2d}: omega(1e4) = {inv_1e4[i]:.3e}   omega(1e6) = {inv_1e6[i]:.3e}"
        f"   wavelength(1e6) = {2 * math.pi / inv_1e6[i]:.3e} tokens"
    )

if plt is not None:
    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.semilogy(inv_1e4, lw=2, color="#2a78d6", label="θ = 10,000")
    ax.semilogy(inv_1e6, lw=2, color="#eb6834", label="θ = 1,000,000 (Qwen3)")
    ax.set_xlabel("pair index i")
    ax.set_ylabel("ω_i (rad / token)")
    ax.grid(alpha=0.25, lw=0.6, which="both")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False)
    ax.set_title("Inverse frequencies: geometric from 1 down to ≈ 1/θ")
    plt.show()

rope = RotaryEmbedding.from_config(CFG)
cos, sin = rope(make_position_ids(4))
print(
    "cos table shape:",
    tuple(cos.shape),
    "| first half == second half:",
    torch.equal(cos[:, :64], cos[:, 64:]),
)

## 4. Worked example: D = 4, θ = 10,000, position 1

$\omega=[1,\;0.01]$. Split-half pairs: $(x_0,x_2)$ uses $\omega_0$, $(x_1,x_3)$ uses $\omega_1$.
With $x=[1,2,3,4]$: $y_0=\cos1-3\sin1$, $y_2=3\cos1+\sin1$,
$y_1=2\cos0.01-4\sin0.01$, $y_3=4\cos0.01+2\sin0.01$.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype=torch.float64).view(1, 1, 1, 4)
inv = compute_inv_freq(4, 10_000.0, dtype=torch.float64)
c, s = rope_cos_sin(torch.tensor([1]), inv)
print("cos          :", c.flatten().tolist())
print("rotate_half x:", rotate_half(x).flatten().tolist())
print("RoPE(x)      :", apply_rope(x, c, s).flatten().tolist())
c1, s1, c2, s2 = math.cos(1), math.sin(1), math.cos(0.01), math.sin(0.01)
print("by hand      :", [c1 - 3 * s1, 2 * c2 - 4 * s2, 3 * c1 + s1, 4 * c2 + 2 * s2])

## 5. Shapes and broadcasting

| tensor | shape | note |
|---|---|---|
| `position_ids` | `[T]` or `[B, T]` | `offset .. offset+T-1` |
| `cos`, `sin` | `[T, D]` or `[B, T, D]` | no head axis |
| broadcast form | `[1 or B, 1, T, D]` | inserted inside `apply_rope` |
| `q` | `[B, Nq, T, D]` | Qwen3-0.6B: Nq = 16 |
| `k` | `[B, Nkv, T, D]` | Nkv = 8 (GQA) |
| `v` | not rotated | positions only affect *which* values are mixed |

In [ ]:
B, T = 2, 5
q = torch.randn(B, CFG.num_attention_heads, T, CFG.head_dim)
k = torch.randn(B, CFG.num_key_value_heads, T, CFG.head_dim)
cos, sin = rope(make_position_ids(T, batch_size=B))
q_rot, k_rot = apply_rope_to_qk(q, k, cos, sin)
print(
    "cos",
    tuple(cos.shape),
    "| q",
    tuple(q.shape),
    "->",
    tuple(q_rot.shape),
    "| k",
    tuple(k.shape),
    "->",
    tuple(k_rot.shape),
)
print(
    "norm preserved (max rel change):",
    ((q_rot.norm(dim=-1) - q.norm(dim=-1)).abs() / q.norm(dim=-1)).max().item(),
)

## 6. Relative positions and attention scores

In [ ]:
torch.manual_seed(0)
d = CFG.head_dim
inv64 = compute_inv_freq(d, CFG.rope_theta, dtype=torch.float64)
qv = torch.randn(1, 1, 1, d, dtype=torch.float64)
kv = qv.clone()  # same content: how does the score change with distance only?


def score(m, n):
    rq = apply_rope(qv, *rope_cos_sin(torch.tensor([m]), inv64))
    rk = apply_rope(kv, *rope_cos_sin(torch.tensor([n]), inv64))
    return (rq * rk).sum().item()


print("score(3, 10)        =", score(3, 10))
print("score(1003, 1010)   =", score(1003, 1010), " (same distance 7)")
print("score(30003, 30010) =", score(30003, 30010))

distances = list(range(0, 2048, 4))
scores = [score(0, n) for n in distances]
if plt is None:
    for n in (0, 1, 4, 16, 64, 256, 1024, 2044):
        print(f"distance {n:5d}: score {score(0, n):9.3f}")
else:
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.plot(distances, scores, lw=1.5, color="#2a78d6")
    ax.axhline(0, color="#9a9a94", lw=0.8)
    ax.set_xlabel("relative distance n - m (tokens)")
    ax.set_ylabel("q · k after RoPE")
    ax.grid(alpha=0.25, lw=0.6)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_title("Identical q and k: score falls and oscillates with distance")
    plt.show()

For identical content, the score is largest at distance 0 and tends to shrink and
oscillate as the fast pairs decorrelate. This is a soft *long-range decay* bias that
the model can override through the content of $q$ and $k$.

## 7. Split-half vs interleaved: same math, different checkpoints

RoFormer pairs adjacent features $(x_{2i},x_{2i+1})$; Qwen3/HF pairs
$(x_i,x_{i+D/2})$ via `rotate_half`. They are equal **after permuting features**,
so a model trained with one layout produces the same scores with the other only if
the `q_proj`/`k_proj` weight rows are permuted accordingly. Loading Qwen3 weights
into an interleaved implementation without that permutation silently gives wrong
attention.

In [ ]:
d = 8
inv = compute_inv_freq(d, 10_000.0, dtype=torch.float64)
pos = make_position_ids(4)
qi = torch.randn(1, 1, 4, d, dtype=torch.float64)
ki = torch.randn(1, 1, 4, d, dtype=torch.float64)
perm = half_layout_permutation(d)
cos, sin = rope_cos_sin(pos, inv)

scores_inter = apply_rope_interleaved(qi, pos, inv) @ apply_rope_interleaved(ki, pos, inv).mT
scores_half_perm = apply_rope(qi[..., perm], cos, sin) @ apply_rope(ki[..., perm], cos, sin).mT
scores_half_noperm = apply_rope(qi, cos, sin) @ apply_rope(ki, cos, sin).mT
print("permutation:", perm.tolist())
print(
    "interleaved vs split-half (permuted) :", (scores_inter - scores_half_perm).abs().max().item()
)
print(
    "interleaved vs split-half (no perm)  :", (scores_inter - scores_half_noperm).abs().max().item()
)

## 8. Positions with a KV cache

During generation the keys of past tokens are rotated **once**, at their own
positions, and stored. A new token at position `offset` (= number of cached tokens)
gets `position_ids = [offset]`. Rotating chunk by chunk with offsets gives exactly
the same tensors as rotating the full sequence at once.

In [ ]:
xseq = torch.randn(1, 2, 10, 16)
r16 = RotaryEmbedding(16)
full = apply_rope(xseq, *r16(make_position_ids(10)))
prefill = apply_rope(xseq[:, :, :7], *r16(make_position_ids(7)))
steps = [
    apply_rope(xseq[:, :, t : t + 1], *r16(make_position_ids(1, offset=t))) for t in range(7, 10)
]
print("incremental == full:", torch.equal(torch.cat([prefill, *steps], dim=2), full))
print(
    "wrong offset (0) for token 9 matches?:",
    torch.equal(apply_rope(xseq[:, :, 9:10], *r16(make_position_ids(1))), full[:, :, 9:10]),
)

## 9. Comparison with Hugging Face and precision

In [ ]:
try:
    import transformers
    from transformers.models.qwen3.modeling_qwen3 import Qwen3RotaryEmbedding, apply_rotary_pos_emb
except ImportError:
    print("transformers not installed -> HF comparison skipped (pip install -e '.[hf]')")
else:
    hf_cfg = transformers.Qwen3Config(
        hidden_size=2048,
        num_attention_heads=16,
        num_key_value_heads=8,
        head_dim=128,
        rope_theta=1e6,
    )
    pos = make_position_ids(64, offset=4000, batch_size=1)
    for dtype in (torch.float32, torch.bfloat16):
        qh = torch.randn(1, 16, 64, 128).to(dtype)
        kh = torch.randn(1, 8, 64, 128).to(dtype)
        hc, hs = Qwen3RotaryEmbedding(hf_cfg)(qh, pos)
        oc, os_ = rope(pos, dtype=dtype)
        hf_out = apply_rotary_pos_emb(qh, kh, hc, hs)
        our_out = apply_rope_to_qk(qh, kh, oc, os_)
        same = torch.equal(hc, oc) and all(
            torch.equal(a, b) for a, b in zip(hf_out, our_out, strict=True)
        )
        print(f"{str(dtype):15s} bit-identical to HF: {same}")

# float32 angles lose precision at large positions (HF does the same):
p = torch.tensor([10, 1000, 32000])
a32 = p[:, None].float() * rope.inv_freq
a64 = p[:, None].double() * inv64
print(
    "max angle error (rad) at positions",
    p.tolist(),
    ":",
    [f"{e:.1e}" for e in (a32.double() - a64).abs().max(dim=-1).values.tolist()],
)

## 10. Gradients

RoPE is a linear, orthogonal map per position: $y=R_m x$, so
$\partial L/\partial x=R_m^\top\,\partial L/\partial y=R_{-m}\,\partial L/\partial y$.
Gradients are rotated back by the same angle and keep their norm.

In [ ]:
xg = torch.randn(1, 2, 1, 16, dtype=torch.float64, requires_grad=True)
c, s = rope_cos_sin(torch.tensor([7]), compute_inv_freq(16, 10_000.0, dtype=torch.float64))
u = torch.randn(1, 2, 1, 16, dtype=torch.float64)
apply_rope(xg, c, s).backward(u)
print("grad == R(-angle) u :", torch.allclose(xg.grad, apply_rope(u, c, -s)))
print("norms               :", xg.grad.norm().item(), u.norm().item())
print(
    "gradcheck           :",
    torch.autograd.gradcheck(
        lambda t: apply_rope(t, c, s), (xg.detach().clone().requires_grad_(),)
    ),
)

## 11. Common implementation errors

- Interleaved pairing with HF weights (section 7), or forgetting to duplicate the
  angles (`cat([freqs, freqs])`) for the split-half layout.
- Rotating `v`, or rotating `q`/`k` *before* QK-Norm (Qwen3: norm first, then RoPE).
- Restarting positions at 0 for every decode step instead of using the cache length.
- Wrong broadcast axis: tables need a head axis (`[B, 1, T, D]`), not a batch axis
  in the head position (HF's `unsqueeze_dim=1` for `[B, N, T, D]`).
- Using `head_dim = hidden_size / num_heads` (64) instead of `head_dim = 128`.
- Wrong theta (10,000 instead of 1,000,000) — outputs look plausible but differ.
- Computing tables in bf16: angles like `30000 * 1.0` are not representable.

## 12. Key lessons and review questions

1. Prove that $\langle R_m q, R_n k\rangle$ depends only on $n-m$.
2. Why does RoPE leave the vector norm unchanged, and why does that matter for attention logits?
3. What happens if two pairs share the same frequency? Why use a geometric sequence?
4. Why is RoPE applied to queries and keys but not values?
5. A model trained with 32,768-token contexts sees position 100,000. Which pairs see
   unfamiliar angles, and why does that hurt extrapolation?
6. What permutation turns an interleaved RoPE checkpoint into a split-half one?